# Project 3: Advanced KG Agent — Demo

This notebook demonstrates an agentic knowledge graph system that can:
- Define and enforce an ontology/schema
- Extract entities with structured output
- Build and query a knowledge graph
- Perform hybrid retrieval (vector + graph)
- Reason across multiple hops
- Handle temporal queries

The agent dynamically decides which retrieval strategy to use based on the question.

In [ ]:
# Setup and imports
import sys
import json
from pathlib import Path
from datetime import datetime
from typing import Optional

from pydantic import BaseModel, Field

# Add projects dir to path for shared imports
sys.path.insert(0, str(Path(".").resolve().parent.parent))

from shared.llm_clients import (
    chat_completion,
    chat_completion_structured,
    get_embedding,
    get_embedding_batch,
)

import networkx as nx
import numpy as np

PROJECT_DIR = Path(".").resolve().parent
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

# Check for Neo4j (optional)
USE_NEO4J = False
try:
    from neo4j import GraphDatabase
    driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "password"))
    driver.verify_connectivity()
    USE_NEO4J = True
    print("Neo4j connected!")
    driver.close()
except Exception:
    print("Neo4j not available — using NetworkX fallback.")

print("Setup complete!")

In [ ]:
# Step 1: Define the ontology/schema
# The schema constrains what entity types and relationship types are valid

ONTOLOGY = {
    "entity_types": {
        "Person": {"properties": ["name", "role", "affiliation"]},
        "Organization": {"properties": ["name", "type", "founded"]},
        "Technology": {"properties": ["name", "category", "version"]},
        "Concept": {"properties": ["name", "domain"]},
        "Event": {"properties": ["name", "date", "location"]},
        "Publication": {"properties": ["title", "year", "authors"]},
    },
    "relationship_types": {
        "WORKS_AT": {"source": "Person", "target": "Organization"},
        "FOUNDED": {"source": "Person", "target": "Organization"},
        "DEVELOPED": {"source": ["Person", "Organization"], "target": "Technology"},
        "USES": {"source": ["Person", "Organization"], "target": "Technology"},
        "PUBLISHED": {"source": "Person", "target": "Publication"},
        "RELATED_TO": {"source": "Concept", "target": "Concept"},
        "PART_OF": {"source": "Technology", "target": "Technology"},
        "OCCURRED_AT": {"source": "Event", "target": "Organization"},
    },
}

print("Ontology defined:")
print(f"  Entity types: {list(ONTOLOGY['entity_types'].keys())}")
print(f"  Relationship types: {list(ONTOLOGY['relationship_types'].keys())}")

# Format for LLM prompt
ontology_str = json.dumps(ONTOLOGY, indent=2)
print(f"\n{ontology_str}")

In [ ]:
# Step 2: Entity extraction with structured output (schema-constrained)

class ExtractedEntity(BaseModel):
    name: str = Field(description="Entity name")
    entity_type: str = Field(description="One of: Person, Organization, Technology, Concept, Event, Publication")
    properties: dict = Field(default_factory=dict, description="Additional properties")

class ExtractedRelationship(BaseModel):
    source: str = Field(description="Source entity name")
    target: str = Field(description="Target entity name")
    relation: str = Field(description="Relationship type from ontology")
    properties: dict = Field(default_factory=dict, description="Edge properties (e.g., since, confidence)")

class SchemaExtraction(BaseModel):
    entities: list[ExtractedEntity]
    relationships: list[ExtractedRelationship]

# Sample text for extraction
sample_text = """OpenAI, founded by Sam Altman and others in 2015, developed GPT-4 and ChatGPT.
GPT-4 is a large language model based on the transformer architecture, which was introduced
in the 2017 paper 'Attention Is All You Need' by researchers at Google Brain.
Microsoft invested $10B in OpenAI in January 2023 and integrated GPT-4 into Bing and Copilot.
Anthropic, founded by former OpenAI researchers Dario and Daniela Amodei in 2021,
developed Claude as a competing AI assistant."""

result = chat_completion_structured(
    prompt=f"""Extract entities and relationships from this text, following this ontology:

{ontology_str}

Text:
{sample_text}""",
    output_schema=SchemaExtraction,
    system="Extract entities and relationships strictly following the provided ontology. Use only the defined entity and relationship types.",
)

print(f"Extracted {len(result.entities)} entities and {len(result.relationships)} relationships\n")

print("Entities:")
for e in result.entities:
    print(f"  [{e.entity_type}] {e.name} {e.properties if e.properties else ''}")

print("\nRelationships:")
for r in result.relationships:
    print(f"  {r.source} --[{r.relation}]--> {r.target} {r.properties if r.properties else ''}")

In [ ]:
# Step 3: Build graph (Neo4j or NetworkX fallback)

G = nx.DiGraph()

# Add entities as nodes
for entity in result.entities:
    props = {"entity_type": entity.entity_type, "label": entity.name}
    props.update(entity.properties)
    G.add_node(entity.name, **props)

# Add relationships as edges
for rel in result.relationships:
    if rel.source in G and rel.target in G:
        props = {"relation": rel.relation}
        props.update(rel.properties)
        G.add_edge(rel.source, rel.target, **props)

print(f"Graph built: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

if USE_NEO4J:
    print("\nAlso writing to Neo4j...")
    driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "password"))
    with driver.session() as session:
        session.run("MATCH (n) DETACH DELETE n")  # Clear existing
        for node, data in G.nodes(data=True):
            etype = data.get("entity_type", "Entity")
            props = {k: str(v) for k, v in data.items()}
            session.run(f"CREATE (n:{etype} $props)", props=props)
        for s, t, data in G.edges(data=True):
            rel_type = data.get("relation", "RELATED")
            session.run(
                f"MATCH (a {{label: $s}}), (b {{label: $t}}) CREATE (a)-[:{rel_type}]->(b)",
                s=s, t=t,
            )
    driver.close()
    print("Written to Neo4j!")
else:
    print("Using NetworkX (Neo4j not available).")

# Display the graph
print("\nNodes:")
for node, data in G.nodes(data=True):
    print(f"  [{data.get('entity_type', '?')}] {node}")
print("\nEdges:")
for s, t, data in G.edges(data=True):
    print(f"  {s} --[{data.get('relation', '?')}]--> {t}")

In [ ]:
# Step 4: Hybrid retrieval — combine vector and graph search

# Embed all entity descriptions for vector search
node_texts = []
node_ids = []
for node, data in G.nodes(data=True):
    text = f"{data.get('entity_type', '')}: {node}. {json.dumps(data)}"
    node_texts.append(text)
    node_ids.append(node)

node_embeddings = np.array(get_embedding_batch(node_texts))
print(f"Embedded {len(node_ids)} nodes")

def hybrid_retrieval(query, top_k=3, hops=2):
    """Combine vector similarity + graph traversal for retrieval."""
    # 1. Vector search to find seed entities
    query_emb = np.array(get_embedding(query))
    similarities = node_embeddings @ query_emb / (
        np.linalg.norm(node_embeddings, axis=1) * np.linalg.norm(query_emb) + 1e-10
    )
    top_idx = np.argsort(similarities)[::-1][:top_k]
    seed_entities = [node_ids[i] for i in top_idx]
    
    # 2. Graph traversal from seed entities
    expanded = set(seed_entities)
    frontier = set(seed_entities)
    for _ in range(hops):
        next_frontier = set()
        for node in frontier:
            if node in G:
                next_frontier |= set(G.successors(node)) | set(G.predecessors(node))
        frontier = next_frontier - expanded
        expanded |= frontier
    
    # 3. Build context
    subgraph = G.subgraph(expanded)
    triples = [f"({s}) --[{d.get('relation', '?')}]--> ({t})" for s, t, d in subgraph.edges(data=True)]
    
    return {
        "seed_entities": seed_entities,
        "expanded_entities": list(expanded),
        "triples": triples,
        "context": "\n".join(triples),
    }

# Demo
result = hybrid_retrieval("What companies are involved in AI development?")
print(f"Seed entities: {result['seed_entities']}")
print(f"Expanded to: {len(result['expanded_entities'])} entities")
print(f"\nRelevant triples:")
for t in result['triples']:
    print(f"  {t}")

In [ ]:
# Step 5: Multi-hop reasoning
# Answer questions that require following multiple edges in the graph

def multi_hop_query(question, max_hops=3):
    """Use chain-of-thought reasoning over the graph to answer multi-hop questions."""
    # Get full graph context
    all_triples = [f"({s}) --[{d.get('relation', '?')}]--> ({t})" for s, t, d in G.edges(data=True)]
    all_entities = [f"{n} ({G.nodes[n].get('entity_type', '?')})" for n in G.nodes()]
    
    answer = chat_completion(
        prompt=f"""Knowledge graph:
Entities: {', '.join(all_entities)}
Relationships:
{chr(10).join(all_triples)}

Question: {question}

Think step by step. Trace the path through the graph to answer the question.
Show your reasoning chain: Entity A --[rel]--> Entity B --[rel]--> Entity C.""",
        system="You are a knowledge graph reasoning agent. Answer by tracing paths through the graph. Show your reasoning chain explicitly.",
    )
    return answer

# Multi-hop questions
questions = [
    "What technology did the founder of OpenAI help create, and who is competing with it?",
    "Trace the path from Google Brain's research to Microsoft's products.",
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {multi_hop_query(q)}")
    print()

In [ ]:
# Step 6: Temporal queries
# Handle questions about time-based relationships

def temporal_query(question):
    """Answer questions that involve temporal reasoning over the graph."""
    # Gather all temporal information
    temporal_facts = []
    for node, data in G.nodes(data=True):
        for key in ["date", "year", "founded", "since"]:
            if key in data:
                temporal_facts.append(f"{node}: {key} = {data[key]}")
    
    for s, t, data in G.edges(data=True):
        for key in ["date", "year", "since"]:
            if key in data:
                temporal_facts.append(f"{s} --[{data.get('relation', '')}]--> {t}: {key} = {data[key]}")
    
    all_triples = [f"({s}) --[{d.get('relation', '?')}]--> ({t})" for s, t, d in G.edges(data=True)]
    
    answer = chat_completion(
        prompt=f"""Knowledge graph:
{chr(10).join(all_triples)}

Temporal facts:
{chr(10).join(temporal_facts) if temporal_facts else 'No explicit dates available.'}

Question: {question}

Answer based on temporal information in the graph. If dates are not explicit, reason from context.""",
        system="You are a temporal reasoning agent. Answer questions about timelines, sequences, and temporal relationships.",
    )
    return answer

# Demo
temporal_questions = [
    "Which organization was founded first — OpenAI or Anthropic?",
    "What is the timeline of major AI developments shown in the graph?",
]

for q in temporal_questions:
    print(f"Q: {q}")
    print(f"A: {temporal_query(q)}")
    print()

In [ ]:
# Step 7: End-to-end agent workflow
# The agent classifies the question and routes to the appropriate strategy

class QueryClassification(BaseModel):
    strategy: str = Field(description="One of: local, global, multi_hop, temporal, hybrid")
    key_entities: list[str] = Field(description="Key entities mentioned in the question")
    reasoning: str = Field(description="Why this strategy was chosen")

def agent_answer(question):
    """Full agent: classify question, select strategy, answer."""
    # Step 1: Classify the question
    classification = chat_completion_structured(
        prompt=f"""Classify this question for a knowledge graph agent:

Question: {question}

Available entities: {', '.join(G.nodes())}

Strategies:
- local: Question about a specific entity and its neighborhood
- global: Broad thematic question about the whole graph
- multi_hop: Requires following a chain of relationships
- temporal: Involves time, dates, or sequences
- hybrid: Combines multiple strategies""",
        output_schema=QueryClassification,
        system="Classify the question and identify the best retrieval strategy.",
    )
    
    print(f"  Strategy: {classification.strategy}")
    print(f"  Key entities: {classification.key_entities}")
    print(f"  Reasoning: {classification.reasoning}")
    
    # Step 2: Execute the chosen strategy
    if classification.strategy == "multi_hop":
        return multi_hop_query(question)
    elif classification.strategy == "temporal":
        return temporal_query(question)
    elif classification.strategy == "local" and classification.key_entities:
        entity = classification.key_entities[0]
        if entity in G:
            # Local search around entity
            result = hybrid_retrieval(question, top_k=1, hops=2)
            return chat_completion(
                prompt=f"Context:\n{result['context']}\n\nQuestion: {question}",
                system="Answer using the graph context.",
            )
    
    # Default: hybrid retrieval
    result = hybrid_retrieval(question, top_k=3, hops=2)
    return chat_completion(
        prompt=f"Context:\n{result['context']}\n\nQuestion: {question}",
        system="Answer using the graph context.",
    )

# Demo the full agent
agent_questions = [
    "Tell me about OpenAI.",
    "How is Google Brain's research connected to Microsoft's products?",
    "What is the timeline of AI company founding?",
    "What are the main themes across all entities?",
]

for q in agent_questions:
    print(f"\n{'='*60}")
    print(f"Q: {q}\n")
    answer = agent_answer(q)
    print(f"\n  Answer: {answer}")

## Summary

This notebook demonstrated the key components of an agentic KG system:

1. **Schema/Ontology**: Defined entity and relationship types to constrain extraction
2. **Structured Extraction**: Used LLM with Pydantic schemas for reliable entity extraction
3. **Graph Construction**: Built the KG in NetworkX (with Neo4j fallback)
4. **Hybrid Retrieval**: Combined vector similarity with graph traversal
5. **Multi-hop Reasoning**: Traced paths through the graph for complex questions
6. **Temporal Queries**: Handled time-based reasoning
7. **Agent Router**: Automatically classified questions and selected the best strategy

### Key Takeaways

- **Schema enforcement** improves extraction quality and consistency
- **Hybrid retrieval** outperforms either vector or graph search alone
- **Query routing** is essential for handling diverse question types
- **Multi-hop reasoning** is where KGs truly shine over flat retrieval

### Next Steps

- Add confidence scores to extracted relationships
- Implement graph-based fact verification
- Build a feedback loop where the agent improves the graph over time
- Scale to larger corpora with incremental graph updates